# Week 41, Day 5: Prefix sums

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w041d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from itertools import accumulate

nums = [3, -2, 5, -1, 2]
prefix = [0, *accumulate(nums)]
print(prefix)
print(prefix[4] - prefix[1], sum(nums[1:4]))
print([prefix[j] - prefix[i] for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if prefix[j] - prefix[i] == 3])

In [ ]:
def count_runs(nums, k):
    seen = {0: 1}               # the empty prefix, before any item
    current = count = 0
    for x in nums:
        current += x
        count += seen.get(current - k, 0)
        seen[current] = seen.get(current, 0) + 1
    return count

print(count_runs([3, -2, 5, -1, 2], 3), count_runs([1, 1, 1], 2), count_runs([1, -1, 0], 0))

In [ ]:
def longest_run(nums, k):
    first = {0: 0}              # prefix value -> earliest number of items giving it
    current, best = 0, 0
    for j, x in enumerate(nums, start=1):
        current += x
        if current - k in first:
            best = max(best, j - first[current - k])
        first.setdefault(current, j)     # keep the earliest only
    return best

print(longest_run([1, -1, 5, -2, 3], 3), longest_run([-2, -1, 2, 1], 1), longest_run([1, 2], 7))

In [ ]:
def divisible_runs(nums, k):
    seen = {0: 1}
    current = count = 0
    for x in nums:
        current = (current + x) % k          # Python's % is never negative for positive k
        count += seen.get(current, 0)
        seen[current] = seen.get(current, 0) + 1
    return count

print(divisible_runs([4, 5, 0, -2, -3, 1], 5))

In [ ]:
def build(grid):
    rows, cols = len(grid), len(grid[0])
    P = [[0] * (cols + 1) for _ in range(rows + 1)]
    for r in range(rows):
        for c in range(cols):
            P[r + 1][c + 1] = grid[r][c] + P[r][c + 1] + P[r + 1][c] - P[r][c]
    return P

def rect(P, r1, c1, r2, c2):
    return P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]

grid = [[3, 0, 1, 4],
        [5, 6, 3, 2],
        [1, 2, 0, 1]]
P = build(grid)
print(rect(P, 1, 1, 3, 3), 6 + 3 + 2 + 0)
print(rect(P, 0, 0, 3, 4), sum(map(sum, grid)))

In [ ]:
from itertools import accumulate

def apply(n, updates):
    diff = [0] * (n + 1)
    for i, j, v in updates:            # add v to positions i .. j-1
        diff[i] += v
        diff[j] -= v
    return list(accumulate(diff[:n]))

# a hall with 8 time slots; each booking needs chairs from slot i up to slot j
bookings = [(0, 3, 20), (2, 6, 35), (5, 8, 10)]
chairs = apply(8, bookings)
print(chairs, max(chairs))

> **Key idea.** Prefix sums with a leading 0 answer any range total as `prefix[j] - prefix[i]`. With a dictionary of earlier prefix sums (seeded with `{0: 1}`), you can count runs with total k in O(n), or find the longest (keeping each prefix's first position), and negatives are fine. Equal remainders of prefix sums mean a run divisible by k. In a grid, rectangle totals take four lookups. A difference array records `+v` at the start and `-v` just after the end of each update, and one running total applies them all.

### Exercise: Range totals

Write `RangeSums(nums)`, which builds prefix sums once, with `total(i, j)` (the sum of `nums[i:j]`, half-open like a slice; `i > j` or anything outside `0 .. len` raises `IndexError`) and `average(i, j)` (raising `ValueError` for an empty range). Every query must be O(1): no `sum` or loops in the methods.

In [ ]:
class RangeSums:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef test_ranges():\n    \"ranges\"\n    r = RangeSums([3, -2, 5, -1, 2])\n    assert r.total(0, 5) == 7 and r.total(1, 4) == 2 and r.total(2, 2) == 0 and r.total(4, 5) == 2 and r.average(0, 4) == 1.25\n    for args in [(3, 2), (0, 6), (-1, 2)]:\n        try:\n            r.total(*args)\n            assert False, args\n        except IndexError:\n            pass\n    try:\n        r.average(1, 1)\n        assert False\n    except ValueError:\n        pass\n    assert RangeSums([]).total(0, 0) == 0\ndef test_o1():\n    \"O(1) queries\"\n    tree = ast.parse(source)\n    for cls in [n for n in tree.body if isinstance(n, ast.ClassDef)]:\n        for f in cls.body:\n            if isinstance(f, ast.FunctionDef) and f.name in (\"total\", \"average\"):\n                assert not any(isinstance(n, (ast.For, ast.While, ast.comprehension)) for n in ast.walk(f)), f.name\n                assert not any(isinstance(n, ast.Name) and n.id == \"sum\" for n in ast.walk(f)), f.name")

### Exercise: Counting runs with a total

Write `count_runs(nums, k)`: how many runs of consecutive items (of length 1 or more) add up to exactly `k`. Numbers may be negative. Use prefix sums and a dictionary, in O(n); the checks use 100,000 numbers.

In [ ]:
def count_runs(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(nums, k):\n    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) == k)\ndef test_small():\n    \"small\"\n    assert count_runs([1, 1, 1], 2) == 2 and count_runs([1, 2, 3], 3) == 2 and count_runs([], 0) == 0 and count_runs([0, 0], 0) == 3\n    rng = random.Random(8)\n    for _ in range(200):\n        nums = [rng.randint(-3, 3) for _ in range(rng.randint(0, 12))]\n        k = rng.randint(-4, 4)\n        assert count_runs(nums, k) == brute(nums, k), (nums, k)\ndef test_big():\n    \"100,000 numbers\"\n    rng = random.Random(1)\n    nums = [rng.randint(-5, 5) for _ in range(100_000)]\n    assert count_runs(nums, 7) > 0")

### Exercise: The longest run with a total

Write `longest_run(nums, k)`: the `(start, end)` of the longest run `nums[start:end]` adding up to exactly `k`, choosing the one that starts **earliest** if several are equally long; `None` if there's none. Numbers may be negative. O(n) with a dictionary of each prefix sum's **first** position.

In [ ]:
def longest_run(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(nums, k):\n    best = None\n    for i in range(len(nums)):\n        for j in range(i + 1, len(nums) + 1):\n            if sum(nums[i:j]) == k and (best is None or j - i > best[1] - best[0]):\n                best = (i, j)\n    return best\ndef test_small():\n    \"small\"\n    assert longest_run([1, -1, 5, -2, 3], 3) == (0, 4) and longest_run([1, 2], 7) is None and longest_run([5], 5) == (0, 1)\n    rng = random.Random(2)\n    for _ in range(300):\n        nums = [rng.randint(-3, 3) for _ in range(rng.randint(0, 12))]\n        k = rng.randint(-3, 3)\n        assert longest_run(nums, k) == brute(nums, k), (nums, k)")

### Exercise: Divisible runs

Write `divisible_runs(nums, k)` for a positive `k`: how many runs of consecutive items have a total divisible by `k`. Numbers may be negative. Count prefix-sum **remainders**, in O(n).

In [ ]:
def divisible_runs(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(nums, k):\n    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) % k == 0)\ndef test_small():\n    \"small\"\n    assert divisible_runs([4, 5, 0, -2, -3, 1], 5) == 7 and divisible_runs([5], 9) == 0 and divisible_runs([], 3) == 0\n    rng = random.Random(3)\n    for _ in range(200):\n        nums = [rng.randint(-6, 6) for _ in range(rng.randint(0, 12))]\n        k = rng.randint(1, 6)\n        assert divisible_runs(nums, k) == brute(nums, k), (nums, k)")

### Exercise: Rectangle totals

Write `GridSums(grid)` for a rectangular list of rows of numbers (at least one row and one column): build 2D prefix sums once, then `total(r1, c1, r2, c2)` returns the total of rows `r1` to `r2 - 1` and columns `c1` to `c2 - 1` (half-open, like slices) in O(1). Also `best_square(k)`: the largest total of any k by k square, as `(total, r, c)` for its top-left corner (earliest row, then column, on a tie).

In [ ]:
class GridSums:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_total():\n    \"rectangles\"\n    g = GridSums([[3, 0, 1, 4], [5, 6, 3, 2], [1, 2, 0, 1]])\n    assert g.total(1, 1, 3, 3) == 11 and g.total(0, 0, 3, 4) == 28 and g.total(0, 3, 1, 4) == 4 and g.total(2, 2, 2, 4) == 0\n    rng = random.Random(4)\n    grid = [[rng.randint(-5, 9) for _ in range(7)] for _ in range(6)]\n    gs = GridSums(grid)\n    for _ in range(100):\n        r1, r2 = sorted(rng.sample(range(7), 2))\n        c1, c2 = sorted(rng.sample(range(8), 2))\n        assert gs.total(r1, c1, r2, c2) == sum(grid[r][c] for r in range(r1, r2) for c in range(c1, c2))\ndef test_square():\n    \"best square\"\n    g = GridSums([[3, 0, 1, 4], [5, 6, 3, 2], [1, 2, 0, 1]])\n    assert g.best_square(2) == (14, 0, 0) and g.best_square(1) == (6, 1, 1) and g.best_square(3) == (21, 0, 0)")

### Exercise: Busiest moment

A room's bookings are `(start, end)` pairs of whole hours from 0 to 24, each covering the hours from `start` up to but not including `end`. Write `busiest_hours(bookings)`: return `(most, hours)`, the largest number of bookings that overlap in any one hour, and a list of the hours (0 to 23) when that happens, in order. Use a difference array. No bookings gives `(0, list(range(24)))`.

In [ ]:
def busiest_hours(bookings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_busy():\n    \"busy hours\"\n    assert busiest_hours([(9, 12), (11, 14), (10, 11), (13, 15)]) == (2, [10, 11, 13])\n    assert busiest_hours([(0, 24)]) == (1, list(range(24))) and busiest_hours([]) == (0, list(range(24)))\n    assert busiest_hours([(8, 9), (8, 9), (8, 10)]) == (3, [8])")

### Bug hunt: No room for the empty start

These prefix sums start with the first item instead of 0, so `prefix[j] - prefix[i]` is off by one item, and a range from the very start can't be expressed: this prints `9` where `sum(nums[0:3])` is `6`. Rebuild `prefix` with a leading 0 so it prints `6 6`.

In [ ]:
nums = [1, 2, 3, 4]
prefix = [nums[0]]
for x in nums[1:]:
    prefix.append(prefix[-1] + x)

def total(i, j):
    return prefix[j] - prefix[i]

print(total(0, 3), sum(nums[0:3]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a leading 0\"\n    assert output.strip() == \"6 6\", output\n    assert total(1, 4) == 9 and total(2, 2) == 0")

### Bug hunt: Runs from the start go missing

This counts runs adding up to `k`, but its dictionary starts empty, so it never counts a run that begins at the first item: it prints `1` for `[2, 1, 1]` with `k = 2`. Seed the dictionary so it prints `2` (`[2]` and `[1, 1]`).

In [ ]:
def count_runs(nums, k):
    seen = {}
    current = count = 0
    for x in nums:
        current += x
        count += seen.get(current - k, 0)
        seen[current] = seen.get(current, 0) + 1
    return count

print(count_runs([2, 1, 1], 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"seed with the empty prefix\"\n    expect_output(\"2\")")

### Bug hunt: An update that never ends

Each update should add `v` only from `i` up to `j - 1`, but the difference array never records the `-v` at `j`, so every update runs on to the end: this prints `[5, 5, 7, 7, 7]`. Fix it so it prints `[5, 5, 2, 0, 0]`.

In [ ]:
from itertools import accumulate

def apply(n, updates):
    diff = [0] * (n + 1)
    for i, j, v in updates:
        diff[i] += v
    return list(accumulate(diff[:n]))

print(apply(5, [(0, 2, 5), (2, 3, 2)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stop at j\"\n    expect_output(\"[5, 5, 2, 0, 0]\")")

### Puzzle: Counting runs with a dictionary

Arrange the lines into a function that counts the runs adding up to `k`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from itertools import accumulate

nums = [2, -1, 3, 0, 4]
prefix = [0, *accumulate(nums)]

p = ___   # prefix
q = ___   # prefix[4] - prefix[2]
r = ___   # -7 % 3
s = ___   # how many lookups a rectangle total takes with 2D prefix sums
t = ___   # the cost of applying u range updates to n items with a difference array: "O(n + u)" or "O(n * u)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [0, 2, 1, 4, 4, 8] and q == 3 and r == 2, \"p, q and r\"\n    assert s == 4 and t == \"O(n + u)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: balanced runs

A list holds only `0`s and `1`s. A run is **balanced** if it has as many 0s as 1s. Write `balanced(bits)`, returning `(count, longest)`: how many balanced runs there are, and the `(start, end)` of the longest one (`bits[start:end]`, earliest start on a tie), or `None` if there are none. One pass, O(n): count each 0 as −1 and each 1 as +1, so a balanced run is a run adding up to 0, and use one dictionary that keeps both how often each prefix value has appeared and where it first appeared. The checks use 100,000 bits.

In [ ]:
def balanced(bits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(bits):\n    count, best = 0, None\n    for i in range(len(bits)):\n        for j in range(i + 1, len(bits) + 1):\n            if bits[i:j].count(0) * 2 == j - i:\n                count += 1\n                if best is None or j - i > best[1] - best[0]:\n                    best = (i, j)\n    return count, best\ndef test_small():\n    \"small\"\n    assert balanced([0, 1]) == (1, (0, 2)) and balanced([1, 1, 1]) == (0, None) and balanced([]) == (0, None)\n    assert balanced([0, 1, 0, 1]) == (4, (0, 4))\n    rng = random.Random(5)\n    for _ in range(300):\n        bits = [rng.randint(0, 1) for _ in range(rng.randint(0, 14))]\n        assert balanced(bits) == brute(bits), bits\ndef test_big():\n    \"100,000 bits\"\n    rng = random.Random(6)\n    bits = [rng.randint(0, 1) for _ in range(100_000)]\n    count, (i, j) = balanced(bits)\n    assert count > 0 and bits[i:j].count(0) * 2 == j - i")

## Warm-up

`prefix[j] - prefix[i]` is the total of `nums[i:j]`: the first `j` items minus the first `i`. Three different runs add up to 3. The last line checks every pair of prefix positions, O(n²); today's main trick finds the same runs in one pass.

## Prefix sums again

With `prefix[0] = 0` and `prefix[i + 1] = prefix[i] + nums[i]`:

```text
nums:          3   -2    5   -1    2
prefix:    0   3    1    6    5    7
           ^ i=0                   ^ i=5
total of nums[i:j]  =  prefix[j] - prefix[i]        (half-open, like slicing)
```

The leading 0 means ranges starting at the beginning need no special case. `itertools.accumulate` builds the running totals for you (add the 0 yourself). Building is O(n); every range total after that is O(1).

When is it worth it? When there are **many** questions about ranges of data that **doesn't change**. If the data changes between questions, the prefix sums would need rebuilding each time; structures for that (Fenwick trees and segment trees) come later in this phase.

## Prefix sums and a dictionary

How many runs of `nums` add up to exactly `k`? A run `nums[i:j]` adds up to `k` when `prefix[j] - prefix[i] == k`, that is, when `prefix[i] == prefix[j] - k`. So walk along, keeping a **count of every prefix sum seen so far**; at each position, the number of runs ending here with total `k` is how many earlier prefixes equal `current - k`:

That's O(n) time and O(n) space, and negative numbers are fine: nothing here depends on totals only going up. Seeding the dictionary with `{0: 1}` counts runs that start at the very beginning.

To find the **longest** such run, remember the **first** position where each prefix sum appeared, instead of a count:

## Remainders

A run's total is divisible by `k` exactly when its two prefix sums leave the **same remainder** when divided by `k`. So count remainders instead of totals:

Python's `%` always gives a result from 0 to k − 1 for a positive `k`, even for negative numbers (`-3 % 5` is 2), which is exactly what's needed here. In some other languages it doesn't, and this is a classic source of bugs when translating.

## Two dimensions

For a grid of numbers, let `P[r][c]` be the total of the rectangle from the top-left corner up to (but not including) row `r` and column `c`. Each entry is built from three neighbours, and any rectangle's total then takes four lookups (adding back the corner that was subtracted twice):

```text
P[r+1][c+1] = grid[r][c] + P[r][c+1] + P[r+1][c] - P[r][c]

total of rows r1..r2-1, columns c1..c2-1  =  P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
```

This is the same idea as an "integral image" in image processing, which makes the sums of many rectangles of pixels quick to compute.

## Difference arrays

Prefix sums turn many **questions** about ranges into O(1) each. Their mirror image turns many **updates** to ranges into O(1) each. To add `v` to every item from `i` to `j - 1`, don't touch them all: record `+v` at `i` and `-v` at `j` in a difference array. One running total at the end rebuilds every value:

With u updates over n positions, that's O(n + u) instead of O(n × u). The same trick counts how many intervals cover each point: the busiest moment for a room, a road or a server.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the difference-array idea and explain it to yourself the other way round: why does a running total of the differences give back the values? Tomorrow is the Forge, where all five of this week's patterns meet in one larger problem.